# **Yolo11m**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATASET      = Path("/content/drive/MyDrive/merged_dataset/merged_dataset")
DATASET_PATH = str(DATASET)
RUNS_PATH    = "/content/runs"

Mounted at /content/drive


In [2]:
yaml_content = f"""train: {DATASET_PATH}/train/images
val:   {DATASET_PATH}/valid/images
test:  {DATASET_PATH}/test/images

nc: 2
names:
  - cigarette
  - cigarette_like_object
"""

(DATASET / "data.yaml").write_text(yaml_content)
print(f"data.yaml written:\n{yaml_content}")

data.yaml written:
train: /content/drive/MyDrive/merged_dataset/merged_dataset/train/images
val:   /content/drive/MyDrive/merged_dataset/merged_dataset/valid/images
test:  /content/drive/MyDrive/merged_dataset/merged_dataset/test/images

nc: 2
names:
  - cigarette
  - cigarette_like_object



In [3]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.3 MB/s eta 0:00:00


# **Running 150 epochs**

In [4]:
!nvidia-smi

Thu May  7 14:42:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Device: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")

model = YOLO("yolo11m.pt")
model.train(
    data         = str(DATASET / "data.yaml"),
    epochs       = 150,
    batch        = 12,
    imgsz        = 640,
    device       = device,
    workers      = 2,
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    cos_lr       = True,
    warmup_epochs = 5,
    patience     = 10,
    cls          = 1.5,
    cache        = True,
    amp          = True,
    mosaic       = 1.0,
    mixup        = 0.15,
    project      = RUNS_PATH,
    name         = "smoking_v4_medium",
    exist_ok     = True,
    save_period  = 10,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: NVIDIA L4
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/merged_dataset/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, h

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79e3db707ce0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [6]:
import shutil

save_path = "/content/drive/MyDrive/smoking_v4_medium_best.pt"
shutil.copy(f"{RUNS_PATH}/smoking_v4_medium/weights/best.pt", save_path)
print(f"best.pt saved to Drive: {save_path}")

best.pt saved to Drive: /content/drive/MyDrive/smoking_v4_medium_best.pt


In [7]:
import shutil

best_src  = f"{RUNS_PATH}/smoking_v4_medium/weights/best.pt"
best_dest = "smoking_v4_medium_best.pt"
shutil.copy(best_src, best_dest)
print(f"best.pt saved to: {best_dest}")

best.pt saved to: smoking_v4_medium_best.pt
